# Colab T4 Benchmark — Tez Aşama 0

Bu notebook bilimsel sonuç üretmez. Amaç, tez deneylerinden önce gerçek NVIDIA T4 ortamını, epoch sürelerini ve GPU bellek kullanımını ölçmektir.

**Önce:** `Runtime > Change runtime type > T4 GPU` seçin.


In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=True)


## GitHub reposunu al

Repo public olduğu için token veya Colab Secret gerekmez. Notebook doğrudan GitHub'daki `main` dalını klonlar ve kullanılan commit SHA'sını kaydeder.


In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/erdemci07/derin-ogrenme-egitim-dinamikleri.git'
repo_dir = Path('/content/derin-ogrenme-egitim-dinamikleri')
shutil.rmtree(repo_dir, ignore_errors=True)

subprocess.run([
    'git', 'clone', '--depth', '1', '--branch', 'main',
    REPO_URL, str(repo_dir)
], check=True)

os.chdir(repo_dir)
commit_sha = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], text=True
).strip()
os.environ['TEZ_GIT_COMMIT'] = commit_sha

print('Repo hazır.')
print('Commit:', commit_sha)


## Sonuçların Google Drive'a kalıcı kaydı


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CIKTI = Path('/content/drive/MyDrive/tez_deney_sonuclari/benchmark_colab_t4')
DRIVE_CIKTI.mkdir(parents=True, exist_ok=True)
print('Çıktı klasörü:', DRIVE_CIKTI)


## Colab bağımlılıkları

Colab'ın kendi CUDA destekli PyTorch kurulumu korunur; bu nedenle `requirements_colab.txt` içinde torch/torchvision yeniden kurulmaz.


In [ ]:
subprocess.run([
    'python', '-m', 'pip', 'install', '-q', '-r', 'requirements_colab.txt'
], check=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA kullanılabilir:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA aktif değil. Runtime ayarından T4 GPU seçin.')


## Benchmarkı çalıştır

1 warmup + 3 ölçüm epochu CNN-v1 ve CIFAR uyarlanmış ResNet-18 için çalıştırılır.


In [ ]:
cikti = DRIVE_CIKTI / 'benchmark_t4.json'
subprocess.run([
    'python', 'benchmark_colab_t4.py',
    '--warmup-epoch', '1',
    '--olcum-epoch', '3',
    '--batch-size', '128',
    '--num-workers', '2',
    '--seed', '42',
    '--cikti', str(cikti),
], check=True, env=os.environ.copy())


## Özeti göster


In [ ]:
import json
veri = json.loads(cikti.read_text(encoding='utf-8'))
print('GPU:', veri['ortam']['gpu'])
print('Commit:', veri['ortam']['git_commit'])
for sonuc in veri['benchmark']:
    print(
        sonuc['model'],
        'ortalama epoch =', round(sonuc['mean_total_seconds_per_epoch'], 2), 'sn',
        '| tepe bellek =', round(sonuc['peak_gpu_memory_mb'], 0), 'MB'
    )
print('\nKaba kapasite tahminleri:')
for k, v in veri['kaba_kapasite_tahminleri'].items():
    if isinstance(v, (int, float)):
        print('-', k, '=', round(v, 2), 'GPU-saat')
print('\nDosya:', cikti)


## Bu aşamadan sonra

`benchmark_t4.json` dosyasındaki gerçek süreler kullanılarak 150, 300 ve maksimum 1.035 koşuluk planın uygulanabilirliği hesaplanacaktır. Bu notebook çıktıları tez bulgusu değildir.
